# Metagenomic Disease Classification Pipeline

In this notebook, I use Logistic Regression Methods for metagenomic disease classification.

The overall process is:

1. Quality control with `fastp`
2. Taxonomic classification with `kraken2`
3. Abundance estimation with `bracken`
4. Building a feature matrix across samples
5. Training a supervised classifier on labeled samples
6. Predicting disease labels for new samples

I organized this notebook so it can be used both as a working draft and as a presentation walkthrough.


## Notes

A few setup notes for this notebook:

- The data here is metagenomic sequencing data.
- The classifier is trained on sample-level features rather than directly on raw FASTQ reads.
- The features used in this version are taxonomic abundances derived from `kraken2` and `bracken`.
- Some paths are placeholders and need to be replaced with the actual paths on my machine before running the full pipeline.


In [1]:
from pathlib import Path
import subprocess
import csv
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

## Configure directories

I use this section to define the main working directories and tool/database paths.


In [2]:
BASE = Path.cwd()
RAW_DIR = BASE / "raw_fastq"
CLEAN_DIR = BASE / "clean_fastq"
KRAKEN_DIR = BASE / "kraken_outputs"
BRACKEN_DIR = BASE / "bracken_outputs"
MODEL_DIR = BASE / "models"

KRAKEN_DB = Path("/path/to/kraken2_db")
READ_LENGTH = 150

for d in [CLEAN_DIR, KRAKEN_DIR, BRACKEN_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BASE, RAW_DIR, CLEAN_DIR, KRAKEN_DIR, BRACKEN_DIR, MODEL_DIR

(PosixPath('/content'),
 PosixPath('/content/raw_fastq'),
 PosixPath('/content/clean_fastq'),
 PosixPath('/content/kraken_outputs'),
 PosixPath('/content/bracken_outputs'),
 PosixPath('/content/models'))

## Example sample manifest

For supervised training, I need multiple labeled samples.

Each row in the manifest includes a sample ID and disease label. If the reads are paired-end, both `r1` and `r2` are included. If the reads are single-end, the workflow can be simplified by removing the paired-end parts.


In [3]:
manifest = pd.DataFrame([
    {"sample_id": "sample_001", "label": "diseaseA", "r1": "sample_001_R1.fastq.gz", "r2": "sample_001_R2.fastq.gz"},
    {"sample_id": "sample_002", "label": "diseaseB", "r1": "sample_002_R1.fastq.gz", "r2": "sample_002_R2.fastq.gz"},
    {"sample_id": "sample_003", "label": "healthy",  "r1": "sample_003_R1.fastq.gz", "r2": "sample_003_R2.fastq.gz"},
])

manifest

,sample_id,label,r1,r2
0,sample_001,diseaseA,sample_001_R1.fastq.gz,sample_001_R2.fastq.gz
1,sample_002,diseaseB,sample_002_R1.fastq.gz,sample_002_R2.fastq.gz
2,sample_003,healthy,sample_003_R1.fastq.gz,sample_003_R2.fastq.gz


## Helper to run shell commands


In [4]:
def run(cmd):
    print("Running:", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True)

## Step 1 - clean reads with fastp


In [5]:
def run_fastp_for_sample(sample_row, threads=8):
    sample_id = sample_row["sample_id"]
    r1_in = RAW_DIR / sample_row["r1"]
    r2_in = RAW_DIR / sample_row["r2"]

    r1_out = CLEAN_DIR / f"{sample_id}_R1.clean.fastq.gz"
    r2_out = CLEAN_DIR / f"{sample_id}_R2.clean.fastq.gz"
    html = CLEAN_DIR / f"{sample_id}.fastp.html"
    json_out = CLEAN_DIR / f"{sample_id}.fastp.json"

    cmd = [
        "fastp",
        "-i", r1_in,
        "-I", r2_in,
        "-o", r1_out,
        "-O", r2_out,
        "-h", html,
        "-j", json_out,
        "-w", str(threads),
        "--detect_adapter_for_pe",
        "-q", "20",
        "-l", "50",
    ]

    run(cmd)
    return r1_out, r2_out

## Step 2 - classify reads with Kraken 2


In [6]:
def run_kraken2_for_sample(sample_id, r1_clean, r2_clean, threads=8, confidence=0.1):
    kraken_out = KRAKEN_DIR / f"{sample_id}.kraken.out"
    kraken_report = KRAKEN_DIR / f"{sample_id}.kraken.report"

    cmd = [
        "kraken2",
        "--db", KRAKEN_DB,
        "--paired",
        "--gzip-compressed",
        "--threads", str(threads),
        "--confidence", str(confidence),
        "--use-names",
        "--report", kraken_report,
        "--output", kraken_out,
        r1_clean,
        r2_clean,
    ]

    run(cmd)
    return kraken_out, kraken_report

## Step 3 - estimate abundances with Bracken


In [7]:
def run_bracken_for_sample(sample_id, kraken_report, read_length=150, level="S", threshold=10):
    bracken_out = BRACKEN_DIR / f"{sample_id}.bracken.tsv"

    cmd = [
        "bracken",
        "-d", KRAKEN_DB,
        "-i", kraken_report,
        "-o", bracken_out,
        "-r", str(read_length),
        "-l", level,
        "-t", str(threshold),
    ]

    run(cmd)
    return bracken_out

## Optional - run preprocessing for all samples

I left this block commented out so I can first verify the paths, database location, and filenames before launching the full preprocessing loop.


In [8]:
# for _, row in manifest.iterrows():
#     sample_id = row["sample_id"]
#     r1_clean, r2_clean = run_fastp_for_sample(row, threads=8)
#     _, kraken_report = run_kraken2_for_sample(sample_id, r1_clean, r2_clean, threads=8, confidence=0.1)
#     run_bracken_for_sample(sample_id, kraken_report, read_length=READ_LENGTH, level="S", threshold=10)

## Step 4 - build a sample-by-taxon feature matrix

This function reads the Bracken outputs and converts them into a machine learning feature table, where each row is a sample and each column is a taxon abundance feature.


In [9]:
def read_bracken_file(bracken_path):
    rows = []
    with open(bracken_path, newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            rows.append(row)
    return rows


def build_feature_matrix(manifest_df, bracken_dir):
    sample_feature_dicts = []

    for _, row in manifest_df.iterrows():
        sample_id = row["sample_id"]
        label = row["label"]
        bracken_path = bracken_dir / f"{sample_id}.bracken.tsv"

        if not bracken_path.exists():
            print(f"Skipping {sample_id} - missing {bracken_path.name}")
            continue

        abundances = read_bracken_file(bracken_path)
        feature_row = {"sample_id": sample_id, "label": label}

        for entry in abundances:
            taxon = entry["name"].strip().replace(" ", "_")
            feature_row[taxon] = float(entry["fraction_total_reads"])

        sample_feature_dicts.append(feature_row)

    feature_df = pd.DataFrame(sample_feature_dicts).fillna(0.0)
    return feature_df

## Build the training table


In [10]:
# feature_df = build_feature_matrix(manifest, BRACKEN_DIR)
# feature_df.to_csv(BASE / "training_features.tsv", sep="\t", index=False)
# feature_df.head()

## Example mock training data for demonstration

I included a small mock dataset here so the notebook is easy to present even before the full real feature table is finalized.


In [11]:
feature_df = pd.DataFrame([
    {"sample_id": "sample_001", "label": "diseaseA", "Bacteroides_fragilis": 0.20, "Escherichia_coli": 0.05, "Faecalibacterium_prausnitzii": 0.01},
    {"sample_id": "sample_002", "label": "diseaseA", "Bacteroides_fragilis": 0.22, "Escherichia_coli": 0.04, "Faecalibacterium_prausnitzii": 0.02},
    {"sample_id": "sample_003", "label": "diseaseB", "Bacteroides_fragilis": 0.03, "Escherichia_coli": 0.28, "Faecalibacterium_prausnitzii": 0.01},
    {"sample_id": "sample_004", "label": "diseaseB", "Bacteroides_fragilis": 0.04, "Escherichia_coli": 0.31, "Faecalibacterium_prausnitzii": 0.00},
    {"sample_id": "sample_005", "label": "healthy",  "Bacteroides_fragilis": 0.12, "Escherichia_coli": 0.02, "Faecalibacterium_prausnitzii": 0.25},
    {"sample_id": "sample_006", "label": "healthy",  "Bacteroides_fragilis": 0.10, "Escherichia_coli": 0.01, "Faecalibacterium_prausnitzii": 0.27},
])

feature_df

,sample_id,label,Bacteroides_fragilis,Escherichia_coli,Faecalibacterium_prausnitzii
0,sample_001,diseaseA,0.20,0.05,0.01
1,sample_002,diseaseA,0.22,0.04,0.02
2,sample_003,diseaseB,0.03,0.28,0.01
3,sample_004,diseaseB,0.04,0.31,0.00
4,sample_005,healthy,0.12,0.02,0.25
5,sample_006,healthy,0.10,0.01,0.27


## Step 5 - train a supervised disease classifier


In [12]:
X = feature_df.drop(columns=["sample_id", "label"])
y = feature_df["label"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, class_weight="balanced")),
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

scores = cross_validate(
    model,
    X,
    y_encoded,
    cv=cv,
    scoring=["accuracy", "f1_macro", "precision_macro", "recall_macro"],
    return_train_score=False,
)

print("Mean accuracy:", scores["test_accuracy"].mean())
print("Mean macro F1:", scores["test_f1_macro"].mean())
print("Mean macro precision:", scores["test_precision_macro"].mean())
print("Mean macro recall:", scores["test_recall_macro"].mean())

ValueError: n_splits=3 cannot be greater than the number of members in each class.

## Fit final model and save it


In [ ]:
model.fit(X, y_encoded)

joblib.dump(model, MODEL_DIR / "metagenomic_disease_classifier.joblib")
joblib.dump(label_encoder, MODEL_DIR / "metagenomic_label_encoder.joblib")

print("Saved model files to:")
print(MODEL_DIR / "metagenomic_disease_classifier.joblib")
print(MODEL_DIR / "metagenomic_label_encoder.joblib")

## Step 6 - predict on a new sample feature vector


In [ ]:
new_sample_df = pd.DataFrame([
    {
        "sample_id": "new_sample_001",
        "Bacteroides_fragilis": 0.05,
        "Escherichia_coli": 0.24,
        "Faecalibacterium_prausnitzii": 0.01,
    }
])

X_new = new_sample_df.drop(columns=["sample_id"])

# Align columns with training features
X_new = X_new.reindex(columns=X.columns, fill_value=0.0)

pred = model.predict(X_new)
proba = model.predict_proba(X_new)
labels = label_encoder.inverse_transform(pred)

for sample_id, label, probs in zip(new_sample_df["sample_id"], labels, proba):
    print("sample:", sample_id)
    print("predicted_label:", label)
    print("class_probabilities:")
    for cls, p in zip(label_encoder.classes_, probs):
        print(f"  {cls}: {p:.4f}")

## Summary

This notebook lays out the basic logic of my supervised metagenomic disease classification workflow:

- clean the raw metagenomic reads
- classify reads taxonomically
- estimate abundance features for each sample
- train a supervised classifier on labeled samples
- predict disease labels for new samples

For a more complete version, the main next steps would be:
- adding more samples
- comparing multiple models such as logistic regression and XGBoost
- doing stronger validation
- reducing leakage and batch effects
- adding feature selection or interpretation
